# Part 2 — Temporal Fraud Detection with A3TGCN

This notebook walks through the full temporal pipeline:
1. Build dynamic graph snapshots from IEEE-CIS data
2. Train the A3TGCN model
3. Train the hybrid XGBoost classifier on GNN embeddings + tabular features
4. Evaluate and visualise results
5. SHAP explainability

In [ ]:
import sys, yaml, torch, numpy as np
from pathlib import Path
sys.path.insert(0, str(Path('..')))

with open('../configs/config.yaml') as f:
    config = yaml.safe_load(f)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

## 1. Build Temporal Dataset

In [ ]:
from src.data.temporal_dataset import build_temporal_dataset, split_temporal

dataset = build_temporal_dataset(
    path_txn='../data/raw/train_transaction.csv',
    path_id ='../data/raw/train_identity.csv',
    n_snapshots=config['data']['n_snapshots'],
    max_rows=config['data']['max_rows'],
)
print(f'Snapshots: {dataset.snapshot_count}')

train_ds, test_ds = split_temporal(dataset, config['temporal']['train_split'])

## 2. Train A3TGCN

In [ ]:
from src.models.temporal_model import A3TGCNFraudModel, TemporalTrainer

node_features = dataset.features[0].shape[1]
model = A3TGCNFraudModel(
    node_features=node_features,
    hidden_dim=config['temporal']['hidden_dim'],
    periods=config['temporal']['periods'],
    dropout=config['temporal']['dropout'],
)
trainer = TemporalTrainer(model, device, config)
trainer.train(train_ds, save_path='../artifacts/temporal_best.pth')

## 3. Evaluate on Future Snapshots

In [ ]:
results = trainer.evaluate(test_ds)
print(f"A3TGCN  AUROC: {results['auroc']:.4f}  AUPRC: {results['auprc']:.4f}")

## 4. Hybrid XGBoost Classifier

In [ ]:
from src.models.hybrid_classifier import HybridFraudDetector

embeddings = trainer.extract_embeddings(dataset, snapshot_idx=-1)
tabular    = dataset.features[-1]
labels     = dataset.targets[-1]

detector = HybridFraudDetector(config)
X = detector.build_features(tabular, embeddings)

split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = labels[:split], labels[split:]

detector.fit(X_train, y_train, X_val=X_test, y_val=y_test)
hybrid_results = detector.evaluate(X_test, y_test)
print(f"Hybrid AUROC: {hybrid_results['auroc']:.4f}  F1: {hybrid_results['f1']:.4f}")

## 5. SHAP Explainability

In [ ]:
from src.explainability.shap_explainer import FraudExplainer

explainer = FraudExplainer(model=detector.model, background_data=X_train)
explainer.plot_global_importance(X_test, save_path='../artifacts/shap_global.png')

top = explainer.top_features(X_test, n=10)
for feat, val in top:
    print(f'  {feat:30s} mean|SHAP| = {val:.4f}')